# 🦙 Phase 2: LoRA Merge + AWQ 4-bit Quantization

Merge Phase 1 LoRA adapters into the base LLaMA 3.2 3B model, then apply AWQ 4-bit quantization for efficient deployment.

**⚠️ Use a fresh Colab notebook with T4 GPU**

## 1 · Install Dependencies

In [ ]:
!pip install -q -U pip
!pip install -q -U \
    transformers \
    peft \
    accelerate \
    bitsandbytes \
    autoawq

print("\n✅ Packages installed! RESTART runtime now:")
print("   Runtime → Restart runtime  (or Ctrl+M .)")

## 2 · Verify GPU

In [ ]:
import torch
print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print(f"CUDA: {torch.version.cuda}")
print("✅ GPU ready!")

## 3 · Mount Google Drive & Load LoRA Adapter

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
ADAPTER_DIR = "/content/drive/MyDrive/llama3_lora_phase1"

# Verify adapter files exist
required = ["adapter_model.safetensors", "adapter_config.json"]
for f in required:
    path = os.path.join(ADAPTER_DIR, f)
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1024**2
        print(f"  ✅ {f} ({size_mb:.1f} MB)")
    else:
        print(f"  ❌ {f} NOT FOUND!")

print(f"\n✅ LoRA adapter found at: {ADAPTER_DIR}")

## 4 · Login to HuggingFace

In [ ]:
from huggingface_hub import login
login()  # Paste your HF token

## 5 · Load Base Model + Merge LoRA Adapters

Load the full-precision base model, attach the trained LoRA adapters, and merge them into the base weights.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

MODEL_ID = "meta-llama/Llama-3.2-3B"
ADAPTER_DIR = "/content/drive/MyDrive/llama3_lora_phase1"

print("📥 Loading base model (full precision for merging)...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)
print("✅ Base model loaded")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
print("✅ Tokenizer loaded")

In [ ]:
# Attach LoRA adapters to base model
print("🔗 Attaching LoRA adapters...")
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
print("✅ LoRA adapters attached")

# Merge LoRA weights into base model
print("🔀 Merging LoRA into base weights...")
merged_model = model.merge_and_unload()
print("✅ LoRA merged! Model is now a standard LLaMA model with fine-tuned weights.")

## 6 · Save Merged Model

In [ ]:
MERGED_DIR = "/content/merged_model"

print("💾 Saving merged model...")
merged_model.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)

!ls -lh {MERGED_DIR}/
print(f"\n✅ Merged model saved to {MERGED_DIR}/")

## 7 · Quick Test (Merged Model)

Verify the merged model works before quantization.

In [ ]:
merged_model.eval()

test_prompts = [
    "### Instruction:\nSummarize the following conversation:\n\n### Input:\nAlice: Hey, are we meeting at 3?\nBob: Yes! Same place.\nAlice: Perfect.\n\n### Response:\n",
    "### Instruction:\nReply as Chandler Bing from Friends:\n\n### Input:\nI just got a promotion!\n\n### Response:\n",
]

print("🧪 Merged Model Inference Test:\n")
for i, prompt in enumerate(test_prompts, 1):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(merged_model.device)
    with torch.no_grad():
        out = merged_model.generate(**inputs, max_new_tokens=100, do_sample=True, temperature=0.7, top_p=0.9)
    response = tokenizer.decode(out[0], skip_special_tokens=True)[len(prompt):].strip()
    print(f"Test {i}: {response[:200]}\n")

## 8 · AWQ 4-bit Quantization

Quantize the merged model to 4-bit using AWQ for efficient deployment.

In [ ]:
# Free GPU memory before quantization
import gc
del merged_model
del base_model
del model
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory freed: {torch.cuda.memory_allocated()/1024**3:.1f} GB used")

In [ ]:
from awq import AutoAWQForCausalLM
from transformers import AutoTokenizer

MERGED_DIR = "/content/merged_model"
AWQ_DIR = "/content/awq_model"

print("📥 Loading merged model for AWQ quantization...")
awq_model = AutoAWQForCausalLM.from_pretrained(MERGED_DIR)
tokenizer = AutoTokenizer.from_pretrained(MERGED_DIR)
print("✅ Model loaded for quantization")

In [ ]:
# AWQ quantization config
quant_config = {
    "zero_point": True,
    "q_group_size": 128,
    "w_bit": 4,
    "version": "GEMM",
}

# Calibration data (representative samples for quantization)
calib_data = [
    "### Instruction:\nSummarize the following scientific document:\n\n### Input:\nDeep learning has achieved remarkable success in natural language processing and computer vision.\n\n### Response:\n",
    "### Instruction:\nReply as Chandler Bing from Friends:\n\n### Input:\nDo you want to go to the gym?\n\n### Response:\n",
    "### Instruction:\nSummarize the following conversation:\n\n### Input:\nAlice: Hey, are we meeting at 3?\nBob: Yes, same place.\n\n### Response:\n",
    "### Instruction:\nReply as Chandler Bing from Friends:\n\n### Input:\nWhat do you do for work?\n\n### Response:\n",
    "### Instruction:\nSummarize the following scientific document:\n\n### Input:\nTransformer architectures have revolutionized NLP by enabling parallel processing of sequences.\n\n### Response:\n",
    "### Instruction:\nReply as Chandler Bing from Friends:\n\n### Input:\nHow are you feeling today?\n\n### Response:\n",
    "### Instruction:\nSummarize the following conversation:\n\n### Input:\nTom: Did you finish the report?\nJane: Almost done, need 10 more minutes.\n\n### Response:\n",
    "### Instruction:\nReply as Chandler Bing from Friends:\n\n### Input:\nWant to grab some pizza?\n\n### Response:\n",
]

print("⚙️  Quantizing to 4-bit AWQ (this takes ~15-30 minutes)...")
awq_model.quantize(tokenizer, quant_config=quant_config, calib_data=calib_data)
print("✅ Quantization complete!")

In [ ]:
# Save quantized model
awq_model.save_quantized(AWQ_DIR)
tokenizer.save_pretrained(AWQ_DIR)

!ls -lh {AWQ_DIR}/
print(f"\n✅ AWQ model saved to {AWQ_DIR}/")

## 9 · Test Quantized Model

In [ ]:
# Load and test the quantized model
from awq import AutoAWQForCausalLM
from transformers import AutoTokenizer
import torch

AWQ_DIR = "/content/awq_model"

print("📥 Loading quantized model...")
quant_model = AutoAWQForCausalLM.from_quantized(AWQ_DIR, fuse_layers=False)
tokenizer = AutoTokenizer.from_pretrained(AWQ_DIR)
print("✅ Quantized model loaded")

# Memory comparison
print(f"\n💾 GPU Memory: {torch.cuda.memory_allocated()/1024**3:.2f} GB (4-bit AWQ)")

In [ ]:
test_prompts = [
    "### Instruction:\nSummarize the following conversation:\n\n### Input:\nAlice: Hey, are we meeting at 3?\nBob: Yes! Same place.\nAlice: Perfect.\n\n### Response:\n",
    "### Instruction:\nReply as Chandler Bing from Friends:\n\n### Input:\nI just got a promotion!\n\n### Response:\n",
    "### Instruction:\nSummarize the following scientific document:\n\n### Input:\nRecent studies have shown that large language models can perform few-shot learning effectively when given appropriate prompts and demonstrations.\n\n### Response:\n",
    "### Instruction:\nReply as Chandler Bing from Friends:\n\n### Input:\nCould you BE any more annoying?\n\n### Response:\n",
]

print("🧪 AWQ Quantized Model — Inference Test:\n")
for i, prompt in enumerate(test_prompts, 1):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to("cuda")
    with torch.no_grad():
        out = quant_model.generate(**inputs, max_new_tokens=100, do_sample=True, temperature=0.7, top_p=0.9)
    response = tokenizer.decode(out[0], skip_special_tokens=True)[len(prompt):].strip()
    print(f"Test {i}: {response[:200]}\n")

## 10 · Save to Google Drive & Download

In [ ]:
# Save to Google Drive
!cp -r /content/awq_model /content/drive/MyDrive/llama3_awq_phase2
!cp -r /content/merged_model /content/drive/MyDrive/llama3_merged_phase2
print("✅ Both models saved to Google Drive:")
print("   📁 MyDrive/llama3_merged_phase2/  (full merged model)")
print("   📁 MyDrive/llama3_awq_phase2/     (4-bit quantized model)")

In [ ]:
# Zip AWQ model for download
!cd /content && zip -r /content/llama3_awq_phase2.zip awq_model/
!ls -lh /content/llama3_awq_phase2.zip

try:
    from google.colab import files
    files.download('/content/llama3_awq_phase2.zip')
    print('📥 Download started!')
except ImportError:
    print('Download llama3_awq_phase2.zip from the Files panel.')

## 📊 Final Summary

| Phase | Output | Location |
|---|---|---|
| Phase 0 | Processed datasets (JSONL) | Local machine |
| Phase 1 | LoRA adapter (18 MB) | Drive: `llama3_lora_phase1/` |
| Phase 2 | Merged model (FP16) | Drive: `llama3_merged_phase2/` |
| Phase 2 | AWQ 4-bit model | Drive: `llama3_awq_phase2/` |

The AWQ 4-bit model is ready for efficient deployment! 🚀